# 2.6 · Two pointers and intervals

*Sequences* · 14 exercises

Walk a sorted array from both ends or with a read and a write pointer, and sort intervals by start to merge and intersect them in one pass.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-2.6) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-2.6) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-2.6)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 2.6.1 · Pair sum in a sorted array  `Core`

**Task.** `nums` is sorted ascending. Write `pair_sum_sorted(nums, target)` that walks a pointer in from each end and returns the indices `[i, j]`, `i < j`, of the first pair it finds summing to `target`, or `[]` when there is none.

**Example.** `[1, 3, 4, 6, 9], 10` -> `[0, 4]`; `[1, 3, 4, 6, 9], 2` -> `[]`

**Big O.** O(N) time, O(1) space: a sum that is too small can only grow by moving the low pointer, one that is too big only shrinks by moving the high pointer.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-1)

In [ ]:
# 2.6.1 Pair sum in a sorted array
def pair_sum_sorted(nums, target):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_1():
    nums = [1, 3, 4, 6, 9]
    assert pair_sum_sorted(nums, 10) == [0, 4]
    assert pair_sum_sorted(nums, 13) == [2, 4]
    assert pair_sum_sorted(nums, 2) == []
    assert pair_sum_sorted([], 0) == []


test_ex_2_6_1()
print('2.6.1 passed')

### 2.6.2 · Squares of a sorted array  `Core`

**Task.** `nums` is sorted ascending and may hold negatives. Write `sorted_squares(nums)` that returns the squares in ascending order without sorting: compare the two ends and fill a new list from the back.

**Example.** `[-4, -1, 0, 3]` -> `[0, 1, 9, 16]`

**Big O.** O(N) time and space: the largest square sits at one of the two ends, so filling the output from the back needs no sort.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-2)

In [ ]:
# 2.6.2 Squares of a sorted array
def sorted_squares(nums):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_2():
    assert sorted_squares([-4, -1, 0, 3]) == [0, 1, 9, 16]
    assert sorted_squares([-3, -2]) == [4, 9]
    assert sorted_squares([2, 5]) == [4, 25]
    assert sorted_squares([]) == []


test_ex_2_6_2()
print('2.6.2 passed')

### 2.6.3 · Smallest difference across two arrays  `Core`

**Task.** Write `smallest_gap(a, b)` that returns the smallest `abs(x - y)` with `x` from `a` and `y` from `b` (both non-empty). Sort copies of both and advance whichever pointer holds the smaller value.

**Example.** `[5, 19, 31], [12, 29, 1]` -> `2`

**Big O.** O(N log N + M log M) time for the sorts, then O(N + M) for the walk: moving the smaller value forward is the only move that can shrink the gap.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-3)

In [ ]:
# 2.6.3 Smallest difference across two arrays
def smallest_gap(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_3():
    assert smallest_gap([5, 19, 31], [12, 29, 1]) == 2
    assert smallest_gap([4], [4]) == 0
    assert smallest_gap([-10, 50], [7]) == 17


test_ex_2_6_3()
print('2.6.3 passed')

### 2.6.4 · Move a value to the end  `Core`

**Task.** Write `move_to_end(nums, value)` that moves every copy of `value` to the end of `nums` in place, keeping the other values in their order, and returns `nums`. Use a read pointer and a write pointer.

**Example.** `[2, 1, 2, 3, 2, 4], 2` -> `[1, 3, 4, 2, 2, 2]`

**Big O.** O(N) time, O(1) space: the write pointer never passes the read pointer, so nothing unread is overwritten.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-4)

In [ ]:
# 2.6.4 Move a value to the end
def move_to_end(nums, value):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_4():
    nums = [2, 1, 2, 3, 2, 4]
    assert move_to_end(nums, 2) is nums
    assert nums == [1, 3, 4, 2, 2, 2]
    assert move_to_end([7, 7], 7) == [7, 7]
    assert move_to_end([1, 2], 9) == [1, 2]
    assert move_to_end([], 1) == []


test_ex_2_6_4()
print('2.6.4 passed')

### 2.6.5 · Monotonic check  `Core`

**Task.** Write `is_monotonic(nums)`: `True` when `nums` never goes down or never goes up (equal neighbours are fine; empty and single-value arrays count).

**Example.** `[1, 2, 2, 5]` -> `True`; `[3, 1, 2]` -> `False`

**Big O.** O(N) time, O(1) space: one pass that rules out each direction as soon as a step goes the other way.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-5)

In [ ]:
# 2.6.5 Monotonic check
def is_monotonic(nums):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_5():
    assert is_monotonic([1, 2, 2, 5])
    assert not is_monotonic([3, 1, 2])
    assert is_monotonic([])
    assert is_monotonic([5, 5, 4, 0])
    assert is_monotonic([1, 1])
    assert not is_monotonic([1, 2, 1])


test_ex_2_6_5()
print('2.6.5 passed')

### 2.6.6 · Merge overlapping intervals  `Core`

**Task.** Each interval is `[start, end]`, in any order. Write `merge_intervals(intervals)` that returns the merged intervals sorted by start; intervals that touch (`end == start`) merge too. Do not change the input.

**Example.** `[[6, 8], [1, 3], [2, 4], [8, 9]]` -> `[[1, 4], [6, 9]]`

**Big O.** O(N log N) time for the sort, O(N) for the output: once sorted by start, an interval can only overlap the last merged one.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-6)

In [ ]:
# 2.6.6 Merge overlapping intervals
def merge_intervals(intervals):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_6():
    given = [[6, 8], [1, 3], [2, 4], [8, 9]]
    assert merge_intervals(given) == [[1, 4], [6, 9]]
    assert given == [[6, 8], [1, 3], [2, 4], [8, 9]]
    assert merge_intervals([]) == []
    assert merge_intervals([[1, 10], [2, 3]]) == [[1, 10]]
    assert merge_intervals([[4, 5], [1, 2]]) == [[1, 2], [4, 5]]


test_ex_2_6_6()
print('2.6.6 passed')

### 2.6.7 · Triples to a target  `Practice`

**Task.** Write `triples_to_target(nums, target)` that returns every distinct triple of values from `nums` (three different positions) summing to `target`. Each triple is ascending and the list is in ascending order. Sort a copy, fix the first value, and two-pointer the rest, skipping repeated values.

**Example.** `[4, -1, 2, 0, -3, 1], 1` -> `[[-3, 0, 4], [-1, 0, 2]]`

**Big O.** O(N^2) time, O(1) extra space beyond the output: fix the smallest value, then a pair sum in the sorted rest is one two-pointer walk.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-7)

In [ ]:
# 2.6.7 Triples to a target
def triples_to_target(nums, target):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_7():
    assert triples_to_target([4, -1, 2, 0, -3, 1], 1) == [[-3, 0, 4], [-1, 0, 2]]
    assert triples_to_target([0, 0, 0, 0], 0) == [[0, 0, 0]]
    assert triples_to_target([1, 2], 3) == []
    assert triples_to_target([2, 2, 1, 1, 0], 4) == [[0, 2, 2], [1, 1, 2]]


test_ex_2_6_7()
print('2.6.7 passed')

### 2.6.8 · Most water between two walls  `Practice`

**Task.** `heights[i]` is a wall at position `i`. Write `most_water(heights)` that returns the largest `(j - i) * min(heights[i], heights[j])` over pairs `i < j` (`0` with fewer than two walls). Start with the widest pair and move the shorter wall inward.

**Example.** `[2, 7, 3, 6, 1, 5]` -> `20`

**Big O.** O(N) time, O(1) space: the shorter wall limits every pair it is part of with a closer partner, so it can be dropped.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-8)

In [ ]:
# 2.6.8 Most water between two walls
def most_water(heights):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_8():
    assert most_water([2, 7, 3, 6, 1, 5]) == 20
    assert most_water([1, 1]) == 1
    assert most_water([4]) == 0
    assert most_water([3, 1, 1, 3]) == 9


test_ex_2_6_8()
print('2.6.8 passed')

### 2.6.9 · Overlaps between two interval lists  `Practice`

**Task.** `a` and `b` are lists of `[start, end]` intervals, each sorted and non-overlapping within itself. Write `interval_overlaps(a, b)` that returns every overlap `[max start, min end]` (a single shared point counts), in order.

**Example.** `[[1, 4], [6, 9]], [[3, 7], [9, 12]]` -> `[[3, 4], [6, 7], [9, 9]]`

**Big O.** O(N + M) time: the interval that ends first cannot overlap anything later in the other list, so its pointer moves on.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-9)

In [ ]:
# 2.6.9 Overlaps between two interval lists
def interval_overlaps(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_9():
    assert interval_overlaps([[1, 4], [6, 9]], [[3, 7], [9, 12]]) == [[3, 4], [6, 7], [9, 9]]
    assert interval_overlaps([[1, 2]], [[3, 4]]) == []
    assert interval_overlaps([], [[1, 5]]) == []
    assert interval_overlaps([[0, 10]], [[1, 2], [4, 5]]) == [[1, 2], [4, 5]]


test_ex_2_6_9()
print('2.6.9 passed')

### 2.6.10 · Shortest stretch to sort  `Practice`

**Task.** Write `unsorted_stretch(nums)` that returns `[start, end]`, the shortest index range which, once sorted, leaves all of `nums` sorted, or `[-1, -1]` when it already is. Sweep left to right tracking the maximum, then right to left tracking the minimum.

**Example.** `[1, 3, 7, 5, 6, 9]` -> `[2, 4]`; `[1, 2, 3]` -> `[-1, -1]`

**Big O.** O(N) time, O(1) space: a value below the running maximum must move, so the last such index is the right edge; the mirror pass finds the left edge.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-10)

In [ ]:
# 2.6.10 Shortest stretch to sort
def unsorted_stretch(nums):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_10():
    assert unsorted_stretch([1, 3, 7, 5, 6, 9]) == [2, 4]
    assert unsorted_stretch([1, 2, 3]) == [-1, -1]
    assert unsorted_stretch([3, 2, 1]) == [0, 2]
    assert unsorted_stretch([1, 2, 4, 3, 3, 5]) == [2, 4]
    assert unsorted_stretch([2, 2]) == [-1, -1]
    assert unsorted_stretch([]) == [-1, -1]


test_ex_2_6_10()
print('2.6.10 passed')

### 2.6.11 · Negatives first, order kept  `Practice`

**Task.** Write `negatives_first(nums)` that moves the negative values before the others (zero counts as non-negative) in place, keeping the order within each group, and returns `nums`. Split the range in half, solve each half, then swap the middle blocks (non-negatives of the left, negatives of the right) with three reversals.

**Example.** `[3, -1, 4, -5, 9, -2]` -> `[-1, -5, -2, 3, 4, 9]`

**Big O.** O(N log N) time, O(log N) stack: each level of the recursion moves every value at most a constant number of times with three reversals.

**Note.** The one-pass swap partition is O(N) but not stable; keeping the order in place costs the log factor.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-11)

In [ ]:
# 2.6.11 Negatives first, order kept
def negatives_first(nums):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_11():
    nums = [3, -1, 4, -5, 9, -2]
    assert negatives_first(nums) is nums
    assert nums == [-1, -5, -2, 3, 4, 9]
    assert negatives_first([]) == []
    assert negatives_first([1, 2]) == [1, 2]
    assert negatives_first([-1, -2]) == [-1, -2]
    assert negatives_first([0, -3, 0, -4, 7]) == [-3, -4, 0, 0, 7]


test_ex_2_6_11()
print('2.6.11 passed')

### 2.6.12 · Next arrangement in order  `Stretch`

**Task.** Write `next_arrangement(nums)` that rearranges `nums` in place into the next larger ordering in dictionary order, or into ascending order when it is already the largest, and returns `nums`. Find the last position whose value is below its right neighbour, swap it with the smallest larger value to its right, then reverse the tail.

**Example.** `[1, 3, 2]` -> `[2, 1, 3]`; `[3, 2, 1]` -> `[1, 2, 3]`

**Big O.** O(N) time, O(1) space: the suffix after the pivot is descending, so one swap and one reversal give the next arrangement.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-12)

In [ ]:
# 2.6.12 Next arrangement in order
def next_arrangement(nums):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_12():
    nums = [1, 3, 2]
    assert next_arrangement(nums) is nums
    assert nums == [2, 1, 3]
    assert next_arrangement([3, 2, 1]) == [1, 2, 3]
    assert next_arrangement([1, 1, 5]) == [1, 5, 1]
    assert next_arrangement([2, 4, 3, 1]) == [3, 1, 2, 4]
    assert next_arrangement([]) == []


test_ex_2_6_12()
print('2.6.12 passed')

### 2.6.13 · Four values to a target  `Stretch`

**Task.** Write `quads_to_target(nums, target)` that returns every distinct group of four values from `nums` (four different positions) summing to `target`. Each group is ascending and the list is in ascending order. Sort a copy, fix two values, and two-pointer the rest, skipping repeats at every level.

**Example.** `[2, -1, 0, 3, -2, 1], 2` -> `[[-2, -1, 2, 3], [-2, 0, 1, 3], [-1, 0, 1, 2]]`

**Big O.** O(N^3) time, O(1) extra space beyond the output: two fixed values, then one two-pointer walk for the last pair.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-13)

In [ ]:
# 2.6.13 Four values to a target
def quads_to_target(nums, target):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_13():
    assert quads_to_target([2, -1, 0, 3, -2, 1], 2) == [[-2, -1, 2, 3], [-2, 0, 1, 3], [-1, 0, 1, 2]]
    assert quads_to_target([1, 1, 1, 1, 1], 4) == [[1, 1, 1, 1]]
    assert quads_to_target([1, 2, 3], 6) == []


test_ex_2_6_13()
print('2.6.13 passed')

### 2.6.14 · Water trapped between bars  `Stretch`

**Task.** `heights[i]` is the height of a bar of width 1. Write `trapped_water(heights)` that returns how many units of water stay between the bars after rain. Walk two pointers inward, always moving the lower side and adding its running maximum minus its height.

**Example.** `[3, 0, 2, 0, 4]` -> `7`

**Big O.** O(N) time, O(1) space: the side with the lower bar is bounded by its own running maximum, because the other side is known to be at least as tall.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-6-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-6-14)

In [ ]:
# 2.6.14 Water trapped between bars
def trapped_water(heights):
    raise NotImplementedError

In [ ]:
def test_ex_2_6_14():
    assert trapped_water([3, 0, 2, 0, 4]) == 7
    assert trapped_water([]) == 0
    assert trapped_water([1, 2, 3]) == 0
    assert trapped_water([5, 1, 1, 5]) == 8
    assert trapped_water([4, 1, 3, 1, 2]) == 3


test_ex_2_6_14()
print('2.6.14 passed')